# <center>**Enhancing LEM-X Imaging with the IROS Reconstruction Pipeline**<center>

## <center>**Sky Reconstruction Efficiency**<center>

In [1]:
from pathlib import Path
from typing import Any, Callable

import numpy as np
import pandas as pd

from bloodmoon.mask import CodedMaskCamera, codedmask
from bloodmoon.io import simulation_files
import darksun as ds
from darksun.data import Log, DataLoader, CatalogueLoader

from IROSrec.handle import config_dirpaths
import imgmaker as mgm
from imgmaker.fns import CameraUnitMap

In [2]:
MASK_FITS: str = "mask_NTHT_20260129_CORRECTED.fits"

SKYFIELD: str = "GalacticCentre"
DATA_FITS: str = "baseline_2-50keV_1ks"

RUN_ID: str = 'GC_rec_detected'

ID_CAMERA_A: str = "cam1a"
ID_CAMERA_B: str = "cam1b"
DATASET: str = "detected"

E_min: float = 2.0  # [keV]
E_max: float = 50.0  # [keV]

In [3]:
MASK_PATH, SIMUL_DATA_PATH, SAVE_PATH = config_dirpaths(
    mask=MASK_FITS,
    skyfield=SKYFIELD,
    simul=DATA_FITS,
    runID=RUN_ID,
)
OUT_RESULTS_PATH = mgm.config_savedata_to()

wfm: CodedMaskCamera = codedmask(MASK_PATH, 2, 1)

filepaths: dict[str, dict[str, Path]] = simulation_files(SIMUL_DATA_PATH)
sdlA = ds.get_data(filepaths[ID_CAMERA_A][DATASET], E_min=E_min, E_max=E_max)
catA = ds.get_catalogue(filepaths[ID_CAMERA_A]['sources'])
sdlB = ds.get_data(filepaths[ID_CAMERA_B][DATASET], E_min=E_min, E_max=E_max)
catB = ds.get_catalogue(filepaths[ID_CAMERA_B]['sources'])

logA, logB = ds.load_database(f"{SAVE_PATH}/IROS_sources_db.fits")

# Loading data...
# Loading completed!


### <center>**Benchmark Tables**<center>

In [4]:
import re

def adjust_Tabfrmt(txt: str) -> str:
    # insert \hline instead of rules (journal guidelines)
    for rule in ('toprule', 'midrule', 'bottomrule'):
        txt = txt.replace(rule, 'hline')
    # shift caption and label at the end (journal guidelines)
    pattern = r"(\\begin\{table\}.*?)(\\caption\{.*?\})\s*(\\label\{.*?\})\s*(\\begin\{tabular\}.*?\\end\{tabular\})"
    replacement = r"\1\4\n\2\n\3"
    txt = re.sub(pattern, replacement, txt, flags=re.DOTALL)
    # convert to onecolumn
    txt = txt.replace('table', 'table*')
    return txt

def sort_by(df: pd.DataFrame, key: str, **kwargs: Any) -> pd.DataFrame:
    """Sort DataFrame wrt input column key."""
    return df.sort_values(by=[key], ascending=False, ignore_index=True, **kwargs)

In [5]:
def gather_cam_data(
    log: Log,
    catalogue: CatalogueLoader,
    sdl: DataLoader,
    camera: CodedMaskCamera,
) -> pd.DataFrame:
    """
    Gathers single camera data from IROS reconstruction database.
    """
    ids = np.array([src.upper() for src in log.log['ID']])
    theta_res_x, theta_res_y = mgm.get_angularcoords_residues(
        log, catalogue, sdl, camera,
    )
    cts = np.array(log.log['fluence'])
    true_cts = mgm.extract_catalogue_fluences(log, catalogue, sdl, camera)
    dmap = {
        log.name: {
            'Source': ids,
            'DthetaX': theta_res_x,
            'DthetaY': theta_res_y,
            'IROS_cts': cts,
            'True_cts': true_cts,
            'Dcts': (cts - true_cts) / np.sqrt(true_cts),
            'SNR': np.array(log.log['snr']),
            # 'thetaX [deg]': np.array(log.log['angle_x']),
            # 'thetaY [deg]': np.array(log.log['angle_y']),
        }
    }
    return pd.DataFrame(dmap)

def get_joint_tab(
    data_camA: pd.DataFrame,
    data_camB: pd.DataFrame,
    unitmap: CameraUnitMap,
) -> pd.DataFrame:
    """Generates a Dataframe with output data from both cameras."""
    compose: Callable = lambda a, b: np.sqrt(a ** 2 + b ** 2)
    dmap = {
        'Source': np.array(data_camA.CAM1A['Source'])[unitmap.idx_a],

        'DthetaX_A': np.array(data_camA.CAM1A['DthetaX'])[unitmap.idx_a],
        'DthetaY_A': np.array(data_camA.CAM1A['DthetaY'])[unitmap.idx_a],
        'TrueCts_A': np.array(data_camA.CAM1A['True_cts'])[unitmap.idx_a],
        'ReconstrCts_A': np.array(data_camA.CAM1A['IROS_cts'])[unitmap.idx_a],
        'Dcts_A': np.array(data_camA.CAM1A['Dcts'])[unitmap.idx_a],

        'DthetaX_B': np.array(data_camB.CAM1B['DthetaX'])[unitmap.idx_b],
        'DthetaY_B': np.array(data_camB.CAM1B['DthetaY'])[unitmap.idx_b],
        'TrueCts_B': np.array(data_camB.CAM1B['True_cts'])[unitmap.idx_b],
        'ReconstrCts_B': np.array(data_camB.CAM1B['IROS_cts'])[unitmap.idx_b],
        'Dcts_B': np.array(data_camB.CAM1B['Dcts'])[unitmap.idx_b],

        'SNR': compose(
            np.array(data_camA.CAM1A['SNR'])[unitmap.idx_a],
            np.array(data_camB.CAM1B['SNR'])[unitmap.idx_b],
        ),
    }
    return pd.DataFrame(dmap)

In [6]:
ds.pixels_angular_resolution(wfm)
cu_map = mgm.get_srcmap_for_unit(logA.log['ID'], logB.log['ID'])

# Table - CAMERA A
data_camA = gather_cam_data(logA, catA, sdlA, wfm)

# Table - CAMERA B
data_camB = gather_cam_data(logB, catB, sdlB, wfm)


Pixel angular resolution at upscaling (x, y): (2, 1)
  - fine direction: 2.1163 arcmin
  - coarse direction: 8.4653 arcmin



Analysing SCOX1:   0%|          | 0/24 [00:00<?, ?it/s]

CATALOG =RXTE-ASM_BeppoSAX-WFC_catalog_2-50keV.fits / Catalog file               [astropy.io.fits.card]
Analysing SCOX1:   0%|          | 0/19 [00:00<?, ?it/s]WARNING: The following header keyword is invalid or follows an unrecognized non-standard convention:
CATALOG =RXTE-ASM_BeppoSAX-WFC_catalog_2-50keV.fits / Catalog file               [astropy.io.fits.card]
Analysing X1624-490: 100%|██████████| 19/19 [00:07<00:00,  2.39it/s]  


In [7]:
unit_data = get_joint_tab(data_camA, data_camB, cu_map)

KWS = {
    'label': 'Table1',
    'caption': 'Testing $`to\\_latex`$ fn.',
    'float_format': "%.1f",
    'column_format': 'l' + 'c' * (len(unit_data.columns) - 2) + 'r',
}
tab = mgm.df2TeXtab(
    df=sort_by(unit_data, 'SNR'),
    adjust_tabfrmt=adjust_Tabfrmt,
    save_to=f'{OUT_RESULTS_PATH}/texTable_Unit_results_{DATASET}_{E_min}-{E_max}.tex',
    overwrite=True,
    **KWS,
)

In [8]:
unit_data

,Source,DthetaX_A,DthetaY_A,TrueCts_A,ReconstrCts_A,Dcts_A,DthetaX_B,DthetaY_B,TrueCts_B,ReconstrCts_B,Dcts_B,SNR
0,CIRX1,-0.591088,8.129261,20819.0,1.885678e+04,-13.599366,0.288593,-0.499063,22600.0,22470.130683,-0.863878,32.111750
1,GROJ1655-40,-0.134610,2.809451,32377.0,3.110936e+04,-7.044932,0.025976,-1.701001,32784.0,31929.359982,-4.720111,28.605267
2,GRS1915+105,0.529230,-0.534516,7731.0,7.000557e+03,-8.307460,-0.657549,-4.963606,7410.0,6931.074546,-5.563637,18.774367
3,GX13+1,0.144337,0.257071,36542.0,4.126304e+04,24.696827,0.091932,5.503171,36893.0,41858.444012,25.851521,40.130321
4,GX17+2,0.210697,-1.210368,72125.0,6.989484e+04,-8.304110,-0.023073,-1.051669,76507.0,76117.082755,-1.409684,62.094273
5,GX3+1,0.104723,0.064041,36018.0,4.229988e+04,33.100121,0.158966,-1.861479,36920.0,35618.066603,-6.775759,33.976992
6,GX340+0,-0.214219,6.510455,45803.0,4.149816e+04,-20.114550,-0.034511,-0.235838,48770.0,51065.087853,10.392569,45.922621
7,GX349+2,0.049902,-0.349435,82159.0,8.229057e+04,0.459032,0.000361,1.456229,83036.0,83893.765431,2.976702,72.836391
8,GX354-0,-0.003926,12.122169,11122.0,1.504626e+04,37.210553,-0.027769,20.984748,11114.0,9757.623210,-12.866048,10.007357
9,GX5-1,0.001472,-0.231254,114965.0,1.147023e+05,-0.774858,0.038208,-0.391822,115614.0,115325.889357,-0.847333,116.588606
